# 12 — Perceptroni dhe ura drejt rrjetave nervore

        **Pyetja udhëheqëse:** Sa larg mund të shkojmë me një ndarës linear që mëson nga gabimet?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të implementohet perceptroni nga zero.
- Të kuptohet rregulli i përditësimit dhe konvergjenca.
- Të analizohet dështimi në të dhëna jo të ndashme linearisht.
- Të kuptohet ura konceptuale drejt rrjetave nervore.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("12 — Perceptroni dhe ura drejt rrjetave nervore")
print("Seed:", RANDOM_STATE)

# 1. Neuroni artificial si ndarës linear — CORE

Perceptroni përdor
$$
z=\mathbf w^T\mathbf x+b,
$$
dhe vendimin
$$
\hat y=\mathrm{sign}(z).
$$
Nëse një mostër $(x_i,y_i)$ klasifikohet gabim, përditësojmë
$$
\mathbf w\leftarrow\mathbf w+\eta y_i\mathbf x_i,\qquad
b\leftarrow b+\eta y_i.
$$

In [ ]:
X0=rng.normal([-1,-1],[0.55,0.55],size=(90,2))
X1=rng.normal([1.2,1.0],[0.55,0.55],size=(90,2))
X=np.vstack([X0,X1]); y=np.r_[np.zeros(len(X0)),np.ones(len(X1))]
fig,ax=plt.subplots()
ax.scatter(X0[:,0],X0[:,1],s=14,label="klasa 0")
ax.scatter(X1[:,0],X1[:,1],s=14,label="klasa 1")
ax.legend(); ax.set_title("Të dhëna linearisht të ndashme")
plt.show()

# 2. Zbatim nga parimet e para — CORE

In [ ]:
def train_perceptron(X,y,eta=1.0,max_epochs=100,seed=42):
    yt=np.where(y>0,1.0,-1.0)
    w=np.zeros(X.shape[1]); b=0.0
    rng_local=np.random.default_rng(seed)
    mistakes=[]
    for _ in range(max_epochs):
        m=0
        for i in rng_local.permutation(len(X)):
            if yt[i]*(X[i]@w+b)<=0:
                w+=eta*yt[i]*X[i]
                b+=eta*yt[i]
                m+=1
        mistakes.append(m)
        if m==0: break
    return w,b,np.array(mistakes)

w,b,mist=train_perceptron(X,y)
w,b,len(mist)

### Perceptroni kanonik nga `src/`

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from src.perceptron.perceptron import PerceptronFromScratch
per_src = PerceptronFromScratch(max_epochs=100, seed=RANDOM_STATE).fit(X, y)
print("Epokat:", per_src.n_iter_, "gabimet e fundit:", per_src.mistakes_per_epoch_[-1])

# 3. Kufiri dhe konvergjenca — CORE

In [ ]:
xx=np.linspace(X[:,0].min()-1,X[:,0].max()+1,200)
yy=-(w[0]*xx+b)/w[1]
fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],c=y,s=14)
ax.plot(xx,yy)
ax.set_title("Kufiri i perceptronit")
plt.show()

fig,ax=plt.subplots()
ax.plot(np.arange(1,len(mist)+1),mist,marker="o")
ax.set_xlabel("Epoka"); ax.set_ylabel("Gabimet")
ax.set_title("Konvergjenca")
plt.show()

# 4. Krahasim me sklearn — CORE

In [ ]:
from sklearn.linear_model import Perceptron
p=Perceptron(random_state=RANDOM_STATE,max_iter=1000,tol=1e-4).fit(X,y)
print("manual:",w,b)
print("sklearn:",p.coef_[0],p.intercept_[0])

# 5. Të dhëna me zhurmë dhe jo të ndashme — CORE

In [ ]:
Xn=X.copy()
yn=y.copy()
flip=rng.choice(len(yn),size=22,replace=False)
yn[flip]=1-yn[flip]
_,_,mist_n=train_perceptron(Xn,yn,max_epochs=60)
fig,ax=plt.subplots()
ax.plot(mist_n)
ax.set_xlabel("Epoka"); ax.set_ylabel("Gabimet")
ax.set_title("Perceptroni nuk konvergon kur ndarja lineare është e pamundur")
plt.show()

# 6. XOR: kufiri themelor — CORE

In [ ]:
Xxor=np.array([[0,0],[0,1],[1,0],[1,1]],float)
yxor=np.array([0,1,1,0])
fig,ax=plt.subplots()
ax.scatter(Xxor[:,0],Xxor[:,1],c=yxor,s=140)
for i,(a,bx) in enumerate(Xxor):
    ax.text(a+0.03,bx+0.03,str(yxor[i]))
ax.set_xlim(-0.3,1.3); ax.set_ylim(-0.3,1.3)
ax.set_title("XOR nuk ndahet me një vijë të vetme")
plt.show()

**Interpretim.** Dështimi në XOR nuk është problem i algoritmit të trajnimit; është kufizim i klasës së modeleve lineare. Kjo motivon shtresa të fshehta dhe transformime jolineare.

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 7. Ura drejt rrjetave nervore — EXTENDED

Një rrjet me shumë shtresa kombinon transformime lineare me funksione aktivizimi jolineare:
$$
h=\phi(Wx+b).
$$
Shtresat e fshehta mund të ndërtojnë përfaqësime ku probleme si XOR bëhen linearisht të ndashme.
Trajnimi i rrjeteve moderne përdor **prapapërhapjen (backpropagation)** dhe variante të zbritjes
stokastike të gradientit. Këtu ndalemi: këto tema i përkasin trajtimit të mëvonshëm në nivel Master.

In [ ]:
# Një transformim i thjeshtë i projektuar me dorë që e bën XOR të dallueshëm
z = (Xxor[:,0] != Xxor[:,1]).astype(float)
fig,ax=plt.subplots()
ax.scatter(z, np.zeros_like(z), c=yxor, s=140)
ax.set_xlabel("veçori e re jolineare: x1 XOR x2")
ax.set_yticks([])
ax.set_title("Një përfaqësim jolinear mund ta zgjidhë problemin")
plt.show()

# 8. Aplikim fizik: ndarje regjimesh — EXTENDED

Në një hapësirë veçorish të ndërtuar nga amplituda dhe frekuenca, perceptroni mund të funksionojë mirë vetëm nëse regjimet ndahen afërsisht me një hiperplan. Ky kusht duhet kontrolluar, jo supozuar.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Ndryshoni learning rate dhe numëroni epokat deri në konvergjencë.
2. Zhvendosni klasat gradualisht derisa të mbivendosen.
3. Krahasoni perceptronin me regresionin logjistik.
4. Shpjegoni pse XOR kërkon jolinearitet.

        ## B. Ushtrim i plotë
        Ndërtoni një dataset me tre veçori ku klasat janë të ndashme dhe verifikoni konvergjencën e perceptronit; më pas shtoni zhurmë etikete dhe analizoni gabimet për epokë.

        ## C. Playground
        Ndryshoni learning rate, seed, mbivendosjen dhe numrin e gabimeve në etiketa.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.